# Ontario Grid Demand Intelligence

**Databricks portfolio edition — Bronze / Silver / Gold + day-ahead forecasting**

This notebook downloads official IESO Hourly Demand Reports, preserves source lineage, validates and types the data, builds Delta tables for analytics, evaluates a leakage-safe 24-hour forecast against a seasonal-naive baseline, and publishes dashboard-ready Gold outputs.

> Source: Independent Electricity System Operator (IESO). Copyright © 2004–present Independent Electricity System Operator, all rights reserved. This educational portfolio project is not affiliated with or endorsed by the IESO.

## Architecture and verification contract

1. **Bronze** — raw report rows plus source URL and ingestion timestamp  
2. **Silver** — typed timestamps, numeric demand fields and duplicate removal  
3. **Gold** — daily KPIs, hourly load profiles, peak hours and a 24-hour forecast  
4. **ML** — chronological train/validation/test split; all rolling features are shifted 24 hours  
5. **Dashboard** — SQL-ready Delta tables in the `portfolio` schema

Run all cells before treating the Databricks results as verified. Local repository metrics are not a substitute for a successful workspace execution.

In [ ]:
%pip install -q scikit-learn requests

In [ ]:
from io import StringIO
import json
import math
import numpy as np
import pandas as pd
import requests
from pyspark.sql import functions as F, types as T
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

spark.sql("CREATE SCHEMA IF NOT EXISTS portfolio")

YEAR_URL = "https://reports-public.ieso.ca/public/Demand/PUB_Demand_{year}.csv"
CURRENT_URL = "https://reports-public.ieso.ca/public/Demand/PUB_Demand.csv"
SOURCE_URLS = [YEAR_URL.format(year=year) for year in range(2022, 2026)] + [CURRENT_URL]
RANDOM_STATE = 42
SOURCE_URLS

## 1. Ingest the public reports

IESO reports begin with a short metadata preamble. The helper below keeps the actual CSV header and adds source lineage before Spark reads the combined snapshot.

In [ ]:
def extract_payload(text: str) -> str:
    lines = text.splitlines()
    header_index = next(i for i, line in enumerate(lines) if line.startswith("Date,Hour,"))
    return "\n".join(lines[header_index:])


frames = []
for url in SOURCE_URLS:
    response = requests.get(url, timeout=45)
    response.raise_for_status()
    current = pd.read_csv(StringIO(extract_payload(response.text)))
    current["source_url"] = url
    frames.append(current)

bronze_pdf = pd.concat(frames, ignore_index=True)
bronze_df = spark.createDataFrame(bronze_pdf)
bronze_df = bronze_df.withColumn("ingested_at", F.current_timestamp())

(bronze_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("portfolio.ontario_grid_bronze"))

display(bronze_df.limit(10))

## 2. Build the validated Silver table

The timestamp uses IESO's hour-ending field as a consistent hourly index. Annual and current reports overlap, so the latest ingested record is retained for each timestamp.

In [ ]:
silver_df = (
    bronze_df
    .select(
        F.to_date(F.col("Date")).alias("date"),
        F.col("Hour").cast("int").alias("hour"),
        F.col("Market Demand").cast("double").alias("market_demand_mw"),
        F.col("Ontario Demand").cast("double").alias("ontario_demand_mw"),
        "source_url",
        "ingested_at",
    )
    .filter(F.col("hour").between(1, 24))
    .filter(F.col("ontario_demand_mw").isNotNull())
    .withColumn(
        "timestamp",
        F.to_timestamp(
            F.from_unixtime(
                F.unix_timestamp(F.col("date").cast("timestamp"))
                + (F.col("hour") - 1) * F.lit(3600)
            )
        ),
    )
)

from pyspark.sql.window import Window
latest = Window.partitionBy("timestamp").orderBy(F.col("ingested_at").desc())
silver_df = (
    silver_df.withColumn("row_number", F.row_number().over(latest))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

(silver_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("portfolio.ontario_grid_silver"))

quality_df = silver_df.agg(
    F.count("*").alias("rows"),
    F.min("timestamp").alias("start_timestamp"),
    F.max("timestamp").alias("end_timestamp"),
    F.sum(F.col("ontario_demand_mw").isNull().cast("int")).alias("missing_demand"),
    F.countDistinct("timestamp").alias("distinct_timestamps"),
)
display(quality_df)

## 3. Publish Gold analytics tables

These marts are intentionally narrow and dashboard-ready: daily operations, weekday/weekend load shapes and the highest observed system peaks.

In [ ]:
daily_df = (
    silver_df.groupBy("date")
    .agg(
        F.avg("ontario_demand_mw").alias("average_demand_mw"),
        F.max("ontario_demand_mw").alias("peak_demand_mw"),
        F.min("ontario_demand_mw").alias("minimum_demand_mw"),
        (F.sum("ontario_demand_mw") / 1000).alias("total_energy_gwh"),
    )
    .withColumn("load_factor", F.col("average_demand_mw") / F.col("peak_demand_mw"))
)

profile_df = (
    silver_df
    .withColumn("day_type", F.when(F.dayofweek("timestamp").isin(1, 7), "Weekend").otherwise("Weekday"))
    .withColumn("hour_of_day", F.hour("timestamp"))
    .groupBy("day_type", "hour_of_day")
    .agg(
        F.avg("ontario_demand_mw").alias("average_demand_mw"),
        F.expr("percentile_approx(ontario_demand_mw, 0.90)").alias("p90_demand_mw"),
    )
)

peak_hours_df = silver_df.orderBy(F.col("ontario_demand_mw").desc()).limit(25)

for table_name, dataframe in {
    "portfolio.ontario_grid_gold_daily": daily_df,
    "portfolio.ontario_grid_gold_hourly_profile": profile_df,
    "portfolio.ontario_grid_gold_peak_hours": peak_hours_df,
}.items():
    (dataframe.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(table_name))

display(daily_df.orderBy(F.col("date").desc()).limit(10))

## 4. Train and evaluate a leakage-safe day-ahead model

The final 60 days form the test set. The preceding 60 days are reserved for interval calibration. A same-hour-last-week forecast is the baseline.

In [ ]:
demand_pdf = (
    silver_df.select("timestamp", "ontario_demand_mw")
    .orderBy("timestamp").toPandas()
)
demand_pdf["timestamp"] = pd.to_datetime(demand_pdf["timestamp"])
ts = demand_pdf["timestamp"]
target = demand_pdf["ontario_demand_mw"].astype(float)

demand_pdf["hour"] = ts.dt.hour
demand_pdf["day_of_week"] = ts.dt.dayofweek
demand_pdf["month"] = ts.dt.month
demand_pdf["day_of_year"] = ts.dt.dayofyear
demand_pdf["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
demand_pdf["hour_sin"] = np.sin(2 * np.pi * demand_pdf["hour"] / 24)
demand_pdf["hour_cos"] = np.cos(2 * np.pi * demand_pdf["hour"] / 24)
demand_pdf["week_sin"] = np.sin(2 * np.pi * demand_pdf["day_of_week"] / 7)
demand_pdf["week_cos"] = np.cos(2 * np.pi * demand_pdf["day_of_week"] / 7)
demand_pdf["lag_24h"] = target.shift(24)
demand_pdf["lag_48h"] = target.shift(48)
demand_pdf["lag_168h"] = target.shift(168)
demand_pdf["rolling_mean_24h_safe"] = target.shift(24).rolling(24).mean()
demand_pdf["rolling_mean_168h_safe"] = target.shift(24).rolling(168).mean()

FEATURES = [
    "hour", "day_of_week", "month", "day_of_year", "is_weekend",
    "hour_sin", "hour_cos", "week_sin", "week_cos", "lag_24h",
    "lag_48h", "lag_168h", "rolling_mean_24h_safe", "rolling_mean_168h_safe",
]
model_frame = demand_pdf.dropna(subset=FEATURES + ["ontario_demand_mw"]).copy()
final_day = model_frame["timestamp"].max().normalize()
test_start = final_day - pd.Timedelta(days=59)
validation_start = test_start - pd.Timedelta(days=60)
train = model_frame[model_frame["timestamp"] < validation_start]
validation = model_frame[(model_frame["timestamp"] >= validation_start) & (model_frame["timestamp"] < test_start)]
test = model_frame[model_frame["timestamp"] >= test_start]

def make_model():
    return HistGradientBoostingRegressor(
        learning_rate=0.06, max_iter=260, max_leaf_nodes=31,
        min_samples_leaf=24, l2_regularization=1.0, random_state=RANDOM_STATE,
    )

validation_model = make_model().fit(train[FEATURES], train["ontario_demand_mw"])
validation_pred = validation_model.predict(validation[FEATURES])
residuals = validation["ontario_demand_mw"].to_numpy() - validation_pred
lower_residual, upper_residual = np.quantile(residuals, [0.05, 0.95])

train_validation = pd.concat([train, validation], ignore_index=True)
model = make_model().fit(train_validation[FEATURES], train_validation["ontario_demand_mw"])
test_pred = model.predict(test[FEATURES])
baseline_pred = test["lag_168h"].to_numpy()

def metrics(actual, predicted):
    return {
        "MAE_MW": float(mean_absolute_error(actual, predicted)),
        "RMSE_MW": float(math.sqrt(mean_squared_error(actual, predicted))),
        "MAPE_percent": float(np.mean(np.abs((np.asarray(actual) - predicted) / np.asarray(actual))) * 100),
        "R2": float(r2_score(actual, predicted)),
    }

evaluation = {
    "model": metrics(test["ontario_demand_mw"], test_pred),
    "seasonal_naive_baseline": metrics(test["ontario_demand_mw"], baseline_pred),
    "train_rows": len(train), "validation_rows": len(validation), "test_rows": len(test),
}
evaluation["MAE_improvement_percent"] = (
    (evaluation["seasonal_naive_baseline"]["MAE_MW"] - evaluation["model"]["MAE_MW"])
    / evaluation["seasonal_naive_baseline"]["MAE_MW"] * 100
)
evaluation

## 5. Generate and publish the next 24-hour forecast

Because the forecast horizon is 24 hours, every lag and rolling statistic below still comes from observed history.

In [ ]:
future_timestamps = pd.date_range(
    demand_pdf["timestamp"].max() + pd.Timedelta(hours=1), periods=24, freq="h"
)
future_base = pd.DataFrame({"timestamp": future_timestamps, "ontario_demand_mw": np.nan})
combined = pd.concat([demand_pdf[["timestamp", "ontario_demand_mw"]], future_base], ignore_index=True)
ts = combined["timestamp"]
target = combined["ontario_demand_mw"].astype(float)
combined["hour"] = ts.dt.hour
combined["day_of_week"] = ts.dt.dayofweek
combined["month"] = ts.dt.month
combined["day_of_year"] = ts.dt.dayofyear
combined["is_weekend"] = (ts.dt.dayofweek >= 5).astype(int)
combined["hour_sin"] = np.sin(2 * np.pi * combined["hour"] / 24)
combined["hour_cos"] = np.cos(2 * np.pi * combined["hour"] / 24)
combined["week_sin"] = np.sin(2 * np.pi * combined["day_of_week"] / 7)
combined["week_cos"] = np.cos(2 * np.pi * combined["day_of_week"] / 7)
combined["lag_24h"] = target.shift(24)
combined["lag_48h"] = target.shift(48)
combined["lag_168h"] = target.shift(168)
combined["rolling_mean_24h_safe"] = target.shift(24).rolling(24).mean()
combined["rolling_mean_168h_safe"] = target.shift(24).rolling(168).mean()
future = combined.tail(24).copy()
future_pred = model.predict(future[FEATURES])

forecast_pdf = pd.DataFrame({
    "timestamp": future["timestamp"],
    "forecast_demand_mw": future_pred,
    "lower_90_mw": future_pred + lower_residual,
    "upper_90_mw": future_pred + upper_residual,
})
forecast_df = spark.createDataFrame(forecast_pdf)
(forecast_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("portfolio.ontario_grid_gold_forecast"))

display(forecast_df.orderBy("timestamp"))

## 6. Dashboard handoff

Create a Databricks SQL dashboard from:

- `portfolio.ontario_grid_gold_daily`
- `portfolio.ontario_grid_gold_hourly_profile`
- `portfolio.ontario_grid_gold_peak_hours`
- `portfolio.ontario_grid_gold_forecast`

Suggested visuals: KPI strip, daily average-versus-peak trend, weekday/weekend hourly profile, next-24-hour forecast with bounds, and top demand hours. The repository's `sql/dashboard_queries.sql` contains ready-to-use queries.